# Practical P5: Error Handling & the logging Module
**Learning Outcome**: Write scripts that fail gracefully and produce readable, structured logs instead of print statement spam.

## Part 1: try-except-finally blocks
When dealing with external networks, rate limits, or file access, operations will eventually fail. We use `try/except` to catch these failures before they crash the app.


In [ ]:
def retrieve_key(dict_obj, key):
    try:
        val = dict_obj[key]
        print(f'Successfully retrieved value: {val}')
    except KeyError as e:
        print(f'Error: Key {e} was not found in the dictionary!')
    finally:
        print('Finished lookup attempt.')

d = {'model': 'gemini'}
retrieve_key(d, 'model')
retrieve_key(d, 'temperature') # This will fail gracefully


## Part 2: The logging Module
In production AI microservices, `print` statements are insufficient because they cannot be easily categorized, filtered, or structured. The standard `logging` library offers 5 priority levels:
1. `DEBUG`: Detailed diagnostic info.
2. `INFO`: Normal operation confirmation.
3. `WARNING`: Indication of unexpected events, but script continues.
4. `ERROR`: Serious problem; some functionality failed.
5. `CRITICAL`: Fatal error; program might stop.

Let's configure a basic logger.


In [ ]:
import logging

# Set up root logger configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler() # Output to notebook/console
    ]
)

logger = logging.getLogger('AI_Development')

# Emit messages
logger.debug('This is a debug message (won't show because level is INFO)')
logger.info('Initializing model connection...')
logger.warning('API response latency is high (1500ms).')
logger.error('Failed to connect to API server. Retrying...')


## Hands-On Exercise
**Task**: Write an API simulating function `call_mock_api(payload)`. If `payload` contains the string `'error'`, raise a `ValueError('Invalid payload schema')`. If `payload` is empty, raise a `ConnectionError('Simulated timeout')`. 
Wrap the calls in a `try/except` block and use `logger.error` to log exceptions, and `logger.info` to record successful calls.


In [ ]:
import time

def call_mock_api(payload):
    if not payload:
        raise ConnectionError('Unable to reach LLM API server. Connection timed out.')
    if 'error' in payload:
        raise ValueError('Invalid request data format.')
    return {'status': 'success', 'response': 'This is your AI output.'}

# TODO: Wrap API call simulation with logging
test_payloads = [
    {'text': 'Hello AI!'},
    {'text': 'Trigger an error text.'},
    {} # Empty, should trigger ConnectionError
]

for p in test_payloads:
    try:
        logger.info(f'Sending payload: {p}')
        response = call_mock_api(p)
        logger.info(f'Response successful: {response}')
    except ConnectionError as ce:
        logger.error(f'Connection failed: {ce}')
    except ValueError as ve:
        logger.error(f'Validation failed: {ve}')
    print('-' * 40)
